# Lab 06 - Gold Layer: Analytics, Optimization & Orchestration (SOLUTION)

In [1]:
displayHTML("""
<div style="background: #1B3A4B; color: #FFFFFF; padding: 20px 25px; border-radius: 8px; margin: 15px 0; border-left: 5px solid #F9A825;">
<h2 style="color: #F9A825; margin-top: 0;">The Final Piece of the Lakehouse Puzzle</h2>
</div>
""")

The Final Piece of the Lakehouse Puzzle

---

### Learning Objectives

By the end of this lab, you will be able to:

1. **Create** business-ready aggregation tables from Silver data
2. **Apply** Delta Lake optimization (OPTIMIZE, ZORDER, VACUUM)
3. **Create** visualizations using Databricks built-in UI
4. **Complete** the orchestration pipeline (add Gold task to Workflow)

---

### Gold Layer Architecture

In [1]:
displayHTML("""
<div style="background: #1B3A4B; padding: 30px; border-radius: 12px; margin: 20px 0; font-family: monospace; color: #FFFFFF;">

<div style="text-align: center; font-size: 18px; font-weight: bold; color: #F9A825; margin-bottom: 20px;">GOLD LAYER - Business-Ready Analytics</div>

<div style="text-align: center;">

<div style="display: inline-block; background: #757575; color: #FFFFFF; padding: 15px 30px; border-radius: 8px; font-weight: bold; font-size: 14px; margin-bottom: 15px; border: 2px solid #9E9E9E;">SILVER LAYER<br/>silver.placements<br/><span style="font-size: 11px;">(Cleaned &amp; Enriched Data)</span></div>

<div style="color: #F9A825; font-size: 24px; margin: 10px 0;">&#9660; &#9660; &#9660;</div>

<div style="display: flex; justify-content: center; gap: 20px; flex-wrap: wrap;">

<div style="background: linear-gradient(135deg, #F9A825, #FFC107); color: #1B1F24; padding: 15px 20px; border-radius: 8px; font-weight: bold; text-align: center; min-width: 180px; border: 2px solid #F9A825;">
PORTFOLIO SUMMARY<br/><span style="font-size: 11px;">GroupBy: valuation_date</span><br/><span style="font-size: 10px;">Total value, positions, averages</span>
</div>

<div style="background: linear-gradient(135deg, #F9A825, #FFC107); color: #1B1F24; padding: 15px 20px; border-radius: 8px; font-weight: bold; text-align: center; min-width: 180px; border: 2px solid #F9A825;">
ENTITY PERFORMANCE<br/><span style="font-size: 11px;">GroupBy: axa_entity_id</span><br/><span style="font-size: 10px;">AUM, rankings, positions</span>
</div>

<div style="background: linear-gradient(135deg, #F9A825, #FFC107); color: #1B1F24; padding: 15px 20px; border-radius: 8px; font-weight: bold; text-align: center; min-width: 180px; border: 2px solid #F9A825;">
ASSET ALLOCATION<br/><span style="font-size: 11px;">GroupBy: asset_class</span><br/><span style="font-size: 10px;">Values, percentages, counts</span>
</div>

</div>

</div>

</div>
""")

GOLD LAYER - Business-Ready Analytics 

 

 SILVER LAYER silver.placements (Cleaned & Enriched Data) 

 ▼ ▼ ▼ 

 

 
PORTFOLIO SUMMARY GroupBy: valuation_date Total value, positions, averages 
 

 
ENTITY PERFORMANCE GroupBy: axa_entity_id AUM, rankings, positions 
 

 
ASSET ALLOCATION GroupBy: asset_class Values, percentages, counts

---

**Pre-requisite:** You must have completed **Lab 05 - Silver Layer** before starting this lab.

---

## Section 1: Configuration

In [1]:
displayHTML("""
<div style="background: #1B3A4B; color: #FFFFFF; padding: 12px 20px; border-radius: 6px; margin: 10px 0;">
Set up the environment, configure ADLS access, and load the Silver layer data.
</div>
""")

Set up the environment, configure ADLS access, and load the Silver layer data.

In [ ]:
# ============================================================
# IMPORTS + USER DATABASE
# ============================================================

from pyspark.sql.functions import (
    col, when, lit, current_timestamp, year, month,
    round as spark_round, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, countDistinct, dense_rank
)
from pyspark.sql.window import Window

# Derive a unique database name per participant
_user = spark.sql("SELECT current_user()").first()[0]
_user_clean = _user.split("@")[0].replace(".", "_").replace("-", "_").lower()
USER_DB = f"training_lab_{_user_clean}"
print(f"Your personal database: {USER_DB}")
print("Libraries imported successfully!")


In [ ]:
# ============================================================
# CONFIGURATION (Hive Metastore, table-based)
# ============================================================

DATABASE_NAME = USER_DB
SILVER_TABLE = f"{DATABASE_NAME}.silver_placements"
GOLD_PORTFOLIO_TABLE = f"{DATABASE_NAME}.gold_portfolio_summary"
GOLD_ENTITY_TABLE = f"{DATABASE_NAME}.gold_entity_performance"
GOLD_ALLOCATION_TABLE = f"{DATABASE_NAME}.gold_asset_allocation"

# Verify Silver table from Lab 05 exists
try:
    silver_count = spark.table(SILVER_TABLE).count()
    print(f"Silver table found: {SILVER_TABLE} ({silver_count} records)")
except Exception as e:
    print(f"ERROR: Silver table not found! Run Lab 05 first.")
    raise Exception("Run Lab 05 (Silver Layer) before starting Lab 06") from e

# Cleanup Gold tables for idempotency
for t in [GOLD_PORTFOLIO_TABLE, GOLD_ENTITY_TABLE, GOLD_ALLOCATION_TABLE]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

print("=" * 60)
print("LAB 06 CONFIGURATION")
print("=" * 60)
print(f"Silver Table:     {SILVER_TABLE}")
print(f"Gold Portfolio:   {GOLD_PORTFOLIO_TABLE}")
print(f"Gold Entity:      {GOLD_ENTITY_TABLE}")
print(f"Gold Allocation:  {GOLD_ALLOCATION_TABLE}")


In [ ]:
# ============================================================
# LOAD SILVER DATA
# ============================================================

df_silver = spark.table(SILVER_TABLE)

print(f"Silver records: {df_silver.count()}")
df_silver.printSchema()
display(df_silver.limit(5))

In [1]:
displayHTML("""
<div style="background: #FFFDE7; border-left: 5px solid #F9A825; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">The Gold layer reads from Silver (clean data) and produces pre-aggregated tables optimized for business questions. This is the <em>consumption</em> layer of the Medallion Architecture.</span>
</div>
""")

Key Takeaway: The Gold layer reads from Silver (clean data) and produces pre-aggregated tables optimized for business questions. This is the consumption layer of the Medallion Architecture.

---

## Section 2: Business Aggregations

### Why Pre-Aggregate?

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-left: 5px solid #FF3621; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">The Gold layer pre-computes metrics so business users get instant answers.</strong><br/><br/>
<span style="color: #1B1F24;">Instead of querying millions of Silver records every time, analysts query pre-aggregated Gold tables that return results in seconds. Think of it as building a summary report that is always up to date.</span>
</div>
""")

The Gold layer pre-computes metrics so business users get instant answers. 
 Instead of querying millions of Silver records every time, analysts query pre-aggregated Gold tables that return results in seconds. Think of it as building a summary report that is always up to date.

We will create **3 Gold tables** from our Silver data:

| Gold Table | GroupBy Column | Business Question |
|---|---|---|
| `portfolio_summary` | `valuation_date` | What is our total portfolio value each day? |
| `entity_performance` | `axa_entity_id` | Which entities manage the most assets? |
| `asset_allocation` | `asset_class` | How are investments distributed across asset types? |

### Exercise 2.1: Portfolio Summary by Valuation Date

Create a daily summary showing total market value, number of positions, and average position value.

**Fill in the blanks** to complete the `groupBy` and aggregation functions.

In [ ]:
# ============================================================
# EXERCISE 2.1 - Portfolio Summary (SOLUTION)
# ============================================================

df_portfolio = df_silver.groupBy(
    "valuation_date"                                # SOLUTION
).agg(
    spark_sum("market_value").alias("total_market_value"),   # SOLUTION: "market_value"
    count("*").alias("total_positions"),                     # SOLUTION: "*"
    avg("market_value").alias("avg_position_value")          # SOLUTION: "market_value"
).withColumn("processed_at", current_timestamp()) \
 .orderBy("valuation_date")

print("Portfolio Summary:")
display(df_portfolio)

In [ ]:
# Write Portfolio Summary to Gold layer
df_portfolio.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_PORTFOLIO_TABLE)
print(f"Portfolio Summary written to: {GOLD_PORTFOLIO_TABLE}")
print(f"Table registered: {GOLD_PORTFOLIO_TABLE}")

In [1]:
displayHTML("""
<div style="background: #FFFDE7; border-left: 5px solid #F9A825; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;"><code>groupBy("valuation_date")</code> collapses all rows sharing the same date into one row, then <code>agg()</code> computes summary metrics for each group. This turns thousands of individual positions into a concise daily report.</span>
</div>
""")

Key Takeaway: groupBy("valuation_date") collapses all rows sharing the same date into one row, then agg() computes summary metrics for each group. This turns thousands of individual positions into a concise daily report.

### Exercise 2.2: Entity Performance by Entity ID

Create a performance table per entity with AUM (Assets Under Management), position count, and a **ranking** using Window functions.

**Fill in the blanks** to complete the aggregation.

In [ ]:
# ============================================================
# EXERCISE 2.2 - Entity Performance (SOLUTION)
# ============================================================

df_entity = df_silver.groupBy(
    "axa_entity_id"                                      # SOLUTION
).agg(
    spark_sum("market_value").alias("total_aum"),
    count("*").alias("position_count"),
    countDistinct("asset_class").alias("unique_assets"),
    avg("market_value").alias("avg_position")
)

# Add ranking by total AUM (descending)
window_spec = Window.orderBy(col("total_aum").desc())

df_entity_ranked = df_entity.withColumn(
    "aum_rank", dense_rank().over(window_spec)
).withColumn("processed_at", current_timestamp())

print("Entity Performance (ranked):")
display(df_entity_ranked.orderBy("aum_rank"))

In [ ]:
# Write Entity Performance to Gold layer
df_entity_ranked.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_ENTITY_TABLE)
print(f"Entity Performance written to: {GOLD_ENTITY_TABLE}")
print(f"Table registered: {GOLD_ENTITY_TABLE}")

In [1]:
displayHTML("""
<div style="background: #FFFDE7; border-left: 5px solid #F9A825; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;"><code>dense_rank().over(Window.orderBy(...))</code> assigns a rank to each entity based on its total AUM. Unlike <code>row_number()</code>, <code>dense_rank()</code> gives the same rank to ties, which is usually what business users expect.</span>
</div>
""")

Key Takeaway: dense_rank().over(Window.orderBy(...)) assigns a rank to each entity based on its total AUM. Unlike row_number() , dense_rank() gives the same rank to ties, which is usually what business users expect.

### Exercise 2.3: Asset Allocation with Percentages

Create an allocation table by asset class, including each class's **percentage of total portfolio**.

**Fill in the blanks** to complete the aggregation and percentage calculation.

In [ ]:
# ============================================================
# EXERCISE 2.3 - Asset Allocation (SOLUTION)
# ============================================================

df_allocation = df_silver.groupBy(
    "asset_class"                                         # SOLUTION
).agg(
    spark_sum("market_value").alias("total_value"),
    count("*").alias("position_count"),
    countDistinct("axa_entity_id").alias("entity_count")
)

# Calculate percentage of total portfolio
total_value = df_allocation.agg(spark_sum("total_value")).collect()[0][0]

df_allocation_pct = df_allocation.withColumn(
    "allocation_pct",
    spark_round((col("total_value") / lit(total_value)) * 100, 2)
).withColumn("processed_at", current_timestamp()) \
 .orderBy(col("total_value").desc())

print(f"Total Portfolio Value: {total_value:,.2f}")
print("\nAsset Allocation:")
display(df_allocation_pct)

In [ ]:
# Write Asset Allocation to Gold layer
df_allocation_pct.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_ALLOCATION_TABLE)
print(f"Asset Allocation written to: {GOLD_ALLOCATION_TABLE}")
print(f"Table registered: {GOLD_ALLOCATION_TABLE}")

In [1]:
displayHTML("""
<div style="background: #FFFDE7; border-left: 5px solid #F9A825; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">Calculating <code>allocation_pct</code> requires knowing the total across all groups. We first compute the total with <code>.agg().collect()</code>, then use <code>lit(total_value)</code> to divide each group's value by the portfolio total. This is a common pattern for percentage-of-total calculations.</span>
</div>
""")

Key Takeaway: Calculating allocation_pct requires knowing the total across all groups. We first compute the total with .agg().collect() , then use lit(total_value) to divide each group's value by the portfolio total. This is a common pattern for percentage-of-total calculations.

---

## Section 3: Delta Lake Optimization

### Why Optimize?

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-left: 5px solid #FF3621; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Small files kill query performance.</strong> <span style="color: #1B1F24;">Delta Lake's <code>OPTIMIZE</code> compacts files, <code>ZORDER</code> co-locates data for faster filters, and <code>VACUUM</code> cleans up old versions to save storage costs.</span>
</div>
""")

Small files kill query performance. Delta Lake's OPTIMIZE compacts files, ZORDER co-locates data for faster filters, and VACUUM cleans up old versions to save storage costs.

In [1]:
displayHTML("""
<div style="background: #F9F7F4; padding: 25px; border-radius: 12px; margin: 20px 0; font-family: monospace; font-size: 13px;">

<div style="display: flex; justify-content: space-around; flex-wrap: wrap; gap: 15px;">

<div style="background: #FFFFFF; padding: 15px; border-radius: 8px; text-align: center; min-width: 200px; border: 2px solid #FF3621;">
<div style="font-weight: bold; color: #FF3621; margin-bottom: 8px;">BEFORE OPTIMIZE</div>
<div style="font-size: 20px;">&#128196; &#128196; &#128196; &#128196; &#128196;</div>
<div style="font-size: 11px; margin-top: 5px; color: #1B1F24;">Many small files (1KB each)</div>
<div style="color: #FF3621; font-weight: bold;">SLOW reads</div>
</div>

<div style="background: #FFFFFF; padding: 15px; border-radius: 8px; text-align: center; min-width: 200px; border: 2px solid #00A972;">
<div style="font-weight: bold; color: #00A972; margin-bottom: 8px;">AFTER OPTIMIZE</div>
<div style="font-size: 20px;">&#128214;</div>
<div style="font-size: 11px; margin-top: 5px; color: #1B1F24;">Fewer large files (compacted)</div>
<div style="color: #00A972; font-weight: bold;">FAST reads</div>
</div>

<div style="background: #FFFFFF; padding: 15px; border-radius: 8px; text-align: center; min-width: 200px; border: 2px solid #1B3A4B;">
<div style="font-weight: bold; color: #1B3A4B; margin-bottom: 8px;">ZORDER</div>
<div style="font-size: 11px; color: #1B1F24;">Before: [A1, B2, A3, B1]</div>
<div style="font-size: 11px; color: #1B1F24;">After:&nbsp; [A1, A3, B1, B2]</div>
<div style="font-size: 11px; margin-top: 5px; color: #1B1F24;">Related data co-located</div>
<div style="color: #1B3A4B; font-weight: bold;">FAST filtered queries</div>
</div>

</div>

</div>
""")

BEFORE OPTIMIZE 
 📄 📄 📄 📄 📄 
 Many small files (1KB each) 
 SLOW reads 
 

 
 AFTER OPTIMIZE 
 📖 
 Fewer large files (compacted) 
 FAST reads 
 

 
 ZORDER 
 Before: [A1, B2, A3, B1] 
 After:  [A1, A3, B1, B2] 
 Related data co-located 
 FAST filtered queries

### Exercise 3.1: Check Table Statistics Before Optimization

Before optimizing, let's see the current state of our Delta tables.

In [ ]:
# Check table details: number of files, size, etc.
print("Portfolio Summary - Table Details:")
display(spark.sql(f"DESCRIBE DETAIL {GOLD_PORTFOLIO_TABLE}"))

In [ ]:
print("Entity Performance - Table Details:")
display(spark.sql(f"DESCRIBE DETAIL {GOLD_ENTITY_TABLE}"))

### Exercise 3.2: OPTIMIZE (File Compaction)

The `OPTIMIZE` command compacts small files into larger ones.

**Fill in the blank** with the correct SQL command.

In [ ]:
# EXERCISE 3.2 - OPTIMIZE

# Check file count BEFORE optimization
detail_before = spark.sql(f"DESCRIBE DETAIL {GOLD_PORTFOLIO_TABLE}").select("numFiles", "sizeInBytes").first()
print(f"BEFORE OPTIMIZE:")
print(f"  Files: {detail_before.numFiles}")
print(f"  Size:  {detail_before.sizeInBytes} bytes")

spark.sql(f"OPTIMIZE {GOLD_PORTFOLIO_TABLE}")

# Check file count AFTER optimization
detail_after = spark.sql(f"DESCRIBE DETAIL {GOLD_PORTFOLIO_TABLE}").select("numFiles", "sizeInBytes").first()
print(f"\nAFTER OPTIMIZE:")
print(f"  Files: {detail_after.numFiles}")
print(f"  Size:  {detail_after.sizeInBytes} bytes")
print(f"\nFile reduction: {detail_before.numFiles} -> {detail_after.numFiles}")


### Exercise 3.3: OPTIMIZE with ZORDER

`ZORDER` co-locates data by a chosen column so that queries filtering on that column skip irrelevant files.

**Fill in the blanks**: the ZORDER keyword and the column to sort by.

In [ ]:
# EXERCISE 3.3 - OPTIMIZE with ZORDER

# Check file count BEFORE
detail_before = spark.sql(f"DESCRIBE DETAIL {GOLD_ENTITY_TABLE}").select("numFiles", "sizeInBytes").first()
print(f"BEFORE OPTIMIZE + ZORDER:")
print(f"  Files: {detail_before.numFiles}")
print(f"  Size:  {detail_before.sizeInBytes} bytes")

spark.sql(f"OPTIMIZE {GOLD_ENTITY_TABLE} ZORDER BY (axa_entity_id)")

# Check file count AFTER
detail_after = spark.sql(f"DESCRIBE DETAIL {GOLD_ENTITY_TABLE}").select("numFiles", "sizeInBytes").first()
print(f"\nAFTER OPTIMIZE + ZORDER:")
print(f"  Files: {detail_after.numFiles}")
print(f"  Size:  {detail_after.sizeInBytes} bytes")
print(f"\nFile reduction: {detail_before.numFiles} -> {detail_after.numFiles}")


### Exercise 3.4: VACUUM (Clean Up Old Files)

`VACUUM` removes files that are no longer referenced by the current version of the table.

**Best practice:** Always run a **DRY RUN** first to see what would be deleted.

In [ ]:
# Step 1: DRY RUN - See what files would be removed
print("VACUUM DRY RUN (files that would be deleted):")
display(
    spark.sql(f"""
        VACUUM {GOLD_PORTFOLIO_TABLE} RETAIN 168 HOURS DRY RUN
    """)
)

In [ ]:
# Step 2: Actual VACUUM (removes old files permanently)
spark.sql(f"""
    VACUUM {GOLD_PORTFOLIO_TABLE} RETAIN 168 HOURS
""")

print("VACUUM completed for Portfolio Summary")
print("Retention: 168 hours (7 days) - older unreferenced files removed")

### Exercise 3.5: Verify Optimization Results

Check the table history to confirm that OPTIMIZE and VACUUM operations were recorded.

In [ ]:
# View the history of operations on the table
print("Portfolio Summary - Operation History:")
display(spark.sql(f"DESCRIBE HISTORY {GOLD_PORTFOLIO_TABLE}"))

In [ ]:
# Verify table details after optimization
print("Portfolio Summary - Details After Optimization:")
display(spark.sql(f"DESCRIBE DETAIL {GOLD_PORTFOLIO_TABLE}"))

In [1]:
displayHTML("""
<div style="background: #FFFDE7; border-left: 5px solid #F9A825; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">Run <code>OPTIMIZE</code> regularly on tables that receive frequent writes. Use <code>ZORDER</code> on columns that appear in <code>WHERE</code> clauses of common queries. Use <code>VACUUM</code> to reclaim storage from old file versions, but keep a safe retention period (default 7 days) so time-travel queries still work.</span>
</div>
""")

Key Takeaway: Run OPTIMIZE regularly on tables that receive frequent writes. Use ZORDER on columns that appear in WHERE clauses of common queries. Use VACUUM to reclaim storage from old file versions, but keep a safe retention period (default 7 days) so time-travel queries still work.

---

## Section 4: Visualization with Databricks UI

### Why Visualize in Databricks?

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-left: 5px solid #FF3621; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Data is only valuable when it tells a story.</strong> <span style="color: #1B1F24;">Databricks built-in visualizations let you create charts directly from query results - no Python libraries needed. Just run a SQL query, then configure a chart with clicks.</span>
</div>
""")

Data is only valuable when it tells a story. Databricks built-in visualizations let you create charts directly from query results - no Python libraries needed. Just run a SQL query, then configure a chart with clicks.

In this section, you will:
1. Run SQL queries on your Gold tables
2. Use the **Databricks UI** to create charts from the results
3. Follow step-by-step instructions to build each visualization

### Exercise 4.1: Bar Chart - Asset Allocation

Run the query below, then follow the instructions to create a **Bar Chart**.

In [ ]:
# Query: Asset Allocation overview
display(spark.sql(f"""
SELECT asset_class, total_value, allocation_pct, position_count
FROM {GOLD_ALLOCATION_TABLE}
ORDER BY total_value DESC
"""))


In [1]:
displayHTML("""
<div style="background: #FFFFFF; border: 2px solid #00A972; padding: 20px; border-radius: 10px; margin: 15px 0;">

<div style="font-weight: bold; font-size: 15px; color: #00A972; margin-bottom: 12px;">Instructions: Create a Bar Chart</div>

<ol style="margin: 0; padding-left: 20px; line-height: 1.8; color: #1B1F24;">
  <li>After running the query above, look at the results table</li>
  <li>Click the <strong>"+"</strong> button next to the table tab (bottom of the cell output)</li>
  <li>Select <strong>"Visualization"</strong></li>
  <li>Choose chart type: <strong>Bar</strong></li>
  <li>Set <strong>X-axis</strong>: <code>asset_class</code></li>
  <li>Set <strong>Y-axis</strong>: <code>total_value</code></li>
  <li>Click <strong>"Save"</strong></li>
</ol>

<div style="margin-top: 10px; font-size: 12px; color: #1B1F24;">You should see a vertical bar chart comparing the total value of each asset class.</div>

</div>
""")

Instructions: Create a Bar Chart 

 
 After running the query above, look at the results table 
 Click the "+" button next to the table tab (bottom of the cell output) 
 Select "Visualization" 
 Choose chart type: Bar 
 Set X-axis : asset_class 
 Set Y-axis : total_value 
 Click "Save" 
 

 You should see a vertical bar chart comparing the total value of each asset class.

### Exercise 4.2: Pie Chart - Asset Distribution

Run the query below, then create a **Pie Chart** to see how the portfolio is distributed.

In [ ]:
# Query: Allocation percentages for pie chart
display(spark.sql(f"""
SELECT asset_class, allocation_pct
FROM {GOLD_ALLOCATION_TABLE}
"""))


In [1]:
displayHTML("""
<div style="background: #FFFFFF; border: 2px solid #00A972; padding: 20px; border-radius: 10px; margin: 15px 0;">

<div style="font-weight: bold; font-size: 15px; color: #00A972; margin-bottom: 12px;">Instructions: Create a Pie Chart</div>

<ol style="margin: 0; padding-left: 20px; line-height: 1.8; color: #1B1F24;">
  <li>Click the <strong>"+"</strong> button next to the results table</li>
  <li>Select <strong>"Visualization"</strong></li>
  <li>Choose chart type: <strong>Pie</strong></li>
  <li>Set <strong>Key</strong> (labels): <code>asset_class</code></li>
  <li>Set <strong>Value</strong> (sizes): <code>allocation_pct</code></li>
  <li>Click <strong>"Save"</strong></li>
</ol>

<div style="margin-top: 10px; font-size: 12px; color: #1B1F24;">The pie chart shows each asset class as a slice proportional to its allocation percentage.</div>

</div>
""")

Instructions: Create a Pie Chart 

 
 Click the "+" button next to the results table 
 Select "Visualization" 
 Choose chart type: Pie 
 Set Key (labels): asset_class 
 Set Value (sizes): allocation_pct 
 Click "Save" 
 

 The pie chart shows each asset class as a slice proportional to its allocation percentage.

### Exercise 4.3: Line Chart - Portfolio Value Over Time

Run the query below, then create a **Line Chart** to see how portfolio value evolves.

In [ ]:
# Query: Portfolio value trend over time
display(spark.sql(f"""
SELECT valuation_date, total_market_value, total_positions
FROM {GOLD_PORTFOLIO_TABLE}
ORDER BY valuation_date
"""))


In [1]:
displayHTML("""
<div style="background: #FFFFFF; border: 2px solid #00A972; padding: 20px; border-radius: 10px; margin: 15px 0;">

<div style="font-weight: bold; font-size: 15px; color: #00A972; margin-bottom: 12px;">Instructions: Create a Line Chart</div>

<ol style="margin: 0; padding-left: 20px; line-height: 1.8; color: #1B1F24;">
  <li>Click the <strong>"+"</strong> button next to the results table</li>
  <li>Select <strong>"Visualization"</strong></li>
  <li>Choose chart type: <strong>Line</strong></li>
  <li>Set <strong>X-axis</strong>: <code>valuation_date</code></li>
  <li>Set <strong>Y-axis</strong>: <code>total_market_value</code></li>
  <li>Click <strong>"Save"</strong></li>
</ol>

<div style="margin-top: 10px; font-size: 12px; color: #1B1F24;">The line chart shows how total portfolio value changes across valuation dates. Look for trends, spikes, or drops.</div>

</div>
""")

Instructions: Create a Line Chart 

 
 Click the "+" button next to the results table 
 Select "Visualization" 
 Choose chart type: Line 
 Set X-axis : valuation_date 
 Set Y-axis : total_market_value 
 Click "Save" 
 

 The line chart shows how total portfolio value changes across valuation dates. Look for trends, spikes, or drops.

### Exercise 4.4: Horizontal Bar Chart - Top 10 Entities

Run the query below, then create a **Horizontal Bar Chart** to compare entity performance.

In [ ]:
# Query: Top 10 entities by AUM
display(spark.sql(f"""
SELECT axa_entity_id, total_aum, aum_rank
FROM {GOLD_ENTITY_TABLE}
WHERE aum_rank <= 10
ORDER BY aum_rank
"""))


In [1]:
displayHTML("""
<div style="background: #FFFFFF; border: 2px solid #00A972; padding: 20px; border-radius: 10px; margin: 15px 0;">

<div style="font-weight: bold; font-size: 15px; color: #00A972; margin-bottom: 12px;">Instructions: Create a Horizontal Bar Chart</div>

<ol style="margin: 0; padding-left: 20px; line-height: 1.8; color: #1B1F24;">
  <li>Click the <strong>"+"</strong> button next to the results table</li>
  <li>Select <strong>"Visualization"</strong></li>
  <li>Choose chart type: <strong>Bar</strong></li>
  <li>Set <strong>X-axis</strong>: <code>total_aum</code></li>
  <li>Set <strong>Y-axis</strong>: <code>axa_entity_id</code></li>
  <li>Toggle <strong>"Horizontal"</strong> orientation (or swap axes depending on your Databricks version)</li>
  <li>Click <strong>"Save"</strong></li>
</ol>

<div style="margin-top: 10px; font-size: 12px; color: #1B1F24;">Horizontal bars make it easy to read entity names while comparing their AUM values side by side.</div>

</div>
""")

Instructions: Create a Horizontal Bar Chart 

 
 Click the "+" button next to the results table 
 Select "Visualization" 
 Choose chart type: Bar 
 Set X-axis : total_aum 
 Set Y-axis : axa_entity_id 
 Toggle "Horizontal" orientation (or swap axes depending on your Databricks version) 
 Click "Save" 
 

 Horizontal bars make it easy to read entity names while comparing their AUM values side by side.

In [1]:
displayHTML("""
<div style="background: #FFFDE7; border-left: 5px solid #F9A825; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">Databricks built-in visualizations are the fastest way to explore data. No code needed - just run a SQL query, click "+" &gt; "Visualization", and configure the chart. For production dashboards, you can pin these charts to a <strong>Databricks SQL Dashboard</strong>.</span>
</div>
""")

Key Takeaway: Databricks built-in visualizations are the fastest way to explore data. No code needed - just run a SQL query, click "+" > "Visualization", and configure the chart. For production dashboards, you can pin these charts to a Databricks SQL Dashboard .

---

## Section 5: Complete the Orchestration Pipeline

### Why Orchestrate?

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-left: 5px solid #FF3621; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Your Medallion Pipeline is only complete when all 3 layers run automatically in sequence.</strong><br/><br/>
<span style="color: #1B1F24;">To run the full Medallion pipeline automatically, create a Databricks Workflow with 3 tasks chained in sequence: Bronze → Silver → Gold. Each task runs one of the labs as a notebook.</span>
</div>
""")

Your Medallion Pipeline is only complete when all 3 layers run automatically in sequence. 
 In Lab 04, you created a Workflow with the Bronze task. Now let's add the Silver and Gold tasks to build the full pipeline.

### Complete 3-Task Pipeline

In [1]:
displayHTML("""
<div style="background: #1B3A4B; padding: 30px; border-radius: 12px; margin: 20px 0; font-family: monospace; color: #FFFFFF;">

<div style="text-align: center; font-size: 16px; font-weight: bold; color: #FF3621; margin-bottom: 20px;">MEDALLION PIPELINE - Databricks Workflow</div>

<div style="text-align: center;">

<div style="display: inline-block; background: #CD7F32; color: #FFFFFF; padding: 12px 25px; border-radius: 8px; font-weight: bold; margin: 5px; border: 2px solid #FFF3E0;">
Task 1<br/>BRONZE<br/><span style="font-size: 10px;">04_Bronze_Layer</span>
</div>

<span style="color: #FF3621; font-size: 24px; vertical-align: middle;"> &#10132; </span>

<div style="display: inline-block; background: #757575; color: #FFFFFF; padding: 12px 25px; border-radius: 8px; font-weight: bold; margin: 5px; border: 2px solid #F5F5F5;">
Task 2<br/>SILVER<br/><span style="font-size: 10px;">05_Silver_Layer</span>
</div>

<span style="color: #FF3621; font-size: 24px; vertical-align: middle;"> &#10132; </span>

<div style="display: inline-block; background: #F9A825; color: #1B1F24; padding: 12px 25px; border-radius: 8px; font-weight: bold; margin: 5px; border: 2px solid #FFFDE7;">
Task 3<br/>GOLD<br/><span style="font-size: 10px;">06_Gold_Layer</span>
</div>

</div>

<div style="text-align: center; margin-top: 15px; color: #F9F7F4; font-size: 12px;">Schedule: Daily at 6:00 AM (Europe/Paris) | CRON: 0 0 6 * * ?</div>

</div>
""")

MEDALLION PIPELINE - Databricks Workflow 

 

 
Task 1 BRONZE 04_Bronze_Layer 
 

 ➔ 

 
Task 2 SILVER 05_Silver_Layer 
 

 ➔ 

 
Task 3 GOLD 06_Gold_Layer 
 

 

 Schedule: Daily at 6:00 AM (Europe/Paris) | CRON: 0 0 6 * * ?

### Step-by-Step: Update Your Databricks Workflow

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border: 2px solid #FF3621; padding: 20px; border-radius: 10px; margin: 15px 0;">

<div style="font-weight: bold; font-size: 15px; color: #FF3621; margin-bottom: 12px;">Instructions: Add Silver and Gold Tasks to Your Pipeline</div>

<ol style="margin: 0; padding-left: 20px; line-height: 2.0; color: #1B1F24;">
  <li>Navigate to <strong>Jobs & Pipelines</strong> in the left sidebar</li>
  <li>Find your existing workflow: <strong>"Medallion_Pipeline"</strong></li>
  <li>Click <strong>"Edit"</strong> (or the workflow name to open it)</li>
  <li><strong>Add Task 2</strong>:
    <ul>
      <li>Task name: <code>silver_transformation</code></li>
      <li>Type: Notebook</li>
      <li>Notebook path: <code>/labs/jour2/05_Silver_Layer</code></li>
      <li>Depends on: <code>bronze_ingestion</code></li>
    </ul>
  </li>
  <li><strong>Add Task 3</strong>:
    <ul>
      <li>Task name: <code>gold_aggregations</code></li>
      <li>Type: Notebook</li>
      <li>Notebook path: <code>/labs/jour2/06_Gold_Layer</code></li>
      <li>Depends on: <code>silver_transformation</code></li>
    </ul>
  </li>
  <li>Click <strong>"Run now"</strong> to test the complete pipeline manually</li>
  <li>Once verified, set the schedule: <strong>Daily at 6 AM</strong> (CRON: <code>0 0 6 * * ?</code>)</li>
</ol>

</div>
""")

Instructions: Add Silver and Gold Tasks to Your Pipeline 

 
 Navigate to Jobs & Pipelines in the left sidebar 
 Find your existing workflow: "Medallion_Pipeline" 
 Click "Edit" (or the workflow name to open it) 
 Add Task 2 :
 
 Task name: silver_transformation 
 Type: Notebook 
 Notebook path: /labs/jour2/05_Silver_Layer 
 Depends on: bronze_ingestion 
 
 
 Add Task 3 :
 
 Task name: gold_aggregations 
 Type: Notebook 
 Notebook path: /labs/jour2/06_Gold_Layer 
 Depends on: silver_transformation 
 
 
 Click "Run now" to test the complete pipeline manually 
 Once verified, set the schedule: Daily at 6 AM (CRON: 0 0 6 * * ? )

In [ ]:
# ============================================================
# REFERENCE: Complete Pipeline Configuration (JSON)
# This shows the full job structure for your Workflow
# ============================================================

import json

full_pipeline = {
    "name": "Medallion_Pipeline",
    "tasks": [
        {
            "task_key": "bronze_ingestion",
            "notebook_task": {
                "notebook_path": "/labs/jour2/04_Bronze_Layer"
            }
        },
        {
            "task_key": "silver_transformation",
            "depends_on": [{"task_key": "bronze_ingestion"}],
            "notebook_task": {
                "notebook_path": "/labs/jour2/05_Silver_Layer"
            }
        },
        {
            "task_key": "gold_aggregations",
            "depends_on": [{"task_key": "silver_transformation"}],
            "notebook_task": {
                "notebook_path": "/labs/jour2/06_Gold_Layer"
            }
        }
    ],
    "schedule": {
        "quartz_cron_expression": "0 0 6 * * ?",
        "timezone_id": "Europe/Paris"
    }
}

print("Complete Pipeline Configuration:")
print(json.dumps(full_pipeline, indent=2))

In [1]:
displayHTML("""
<div style="background: #FFFDE7; border-left: 5px solid #F9A825; padding: 15px 20px; border-radius: 8px; margin: 15px 0;">
<strong style="color: #1B1F24;">Key Takeaway:</strong> <span style="color: #1B1F24;">A production Medallion pipeline has 3 tasks chained with dependencies: Bronze &#10132; Silver &#10132; Gold. Databricks Workflows handles scheduling, retries, and alerting. If any task fails, downstream tasks are skipped automatically.</span>
</div>
""")

Key Takeaway: A production Medallion pipeline has 3 tasks chained with dependencies: Bronze ➔ Silver ➔ Gold. Databricks Workflows handles scheduling, retries, and alerting. If any task fails, downstream tasks are skipped automatically.

---

## What You Learned

In [1]:
displayHTML("""
<div style="background: #1B3A4B; padding: 30px; border-radius: 12px; margin: 20px 0; color: #FFFFFF;">

<div style="text-align: center; font-size: 20px; font-weight: bold; color: #F9A825; margin-bottom: 25px;">Lab 06 - Complete Summary</div>

<table style="width: 100%; border-collapse: collapse; font-size: 14px;">
<tr style="border-bottom: 2px solid #F9A825;">
  <th style="text-align: left; padding: 12px; color: #F9A825; width: 35%; font-size: 15px;">Concept</th>
  <th style="text-align: left; padding: 12px; color: #F9F7F4; font-size: 15px;">What You Learned</th>
</tr>
<tr style="border-bottom: 1px solid rgba(255,255,255,0.15);">
  <td style="padding: 12px; color: #F9A825; font-weight: bold;">Gold Layer</td>
  <td style="padding: 12px; color: #FFFFFF;">Pre-computed business metrics for fast analytics (groupBy + agg)</td>
</tr>
<tr style="border-bottom: 1px solid rgba(255,255,255,0.15);">
  <td style="padding: 12px; color: #F9A825; font-weight: bold;">OPTIMIZE</td>
  <td style="padding: 12px; color: #FFFFFF;">Compact small files into larger ones for better read performance</td>
</tr>
<tr style="border-bottom: 1px solid rgba(255,255,255,0.15);">
  <td style="padding: 12px; color: #F9A825; font-weight: bold;">ZORDER</td>
  <td style="padding: 12px; color: #FFFFFF;">Co-locate data by a column so filtered queries skip irrelevant files</td>
</tr>
<tr style="border-bottom: 1px solid rgba(255,255,255,0.15);">
  <td style="padding: 12px; color: #F9A825; font-weight: bold;">VACUUM</td>
  <td style="padding: 12px; color: #FFFFFF;">Clean up old, unreferenced files to save storage costs</td>
</tr>
<tr style="border-bottom: 1px solid rgba(255,255,255,0.15);">
  <td style="padding: 12px; color: #F9A825; font-weight: bold;">Databricks Visualizations</td>
  <td style="padding: 12px; color: #FFFFFF;">Create bar, pie, line charts from query results using the built-in UI</td>
</tr>
<tr>
  <td style="padding: 12px; color: #F9A825; font-weight: bold;">Pipeline Orchestration</td>
  <td style="padding: 12px; color: #FFFFFF;">Automate Bronze &#10132; Silver &#10132; Gold with Databricks Workflows</td>
</tr>
</table>

</div>
""")

Concept,What You Learned
Gold Layer,Pre-computed business metrics for fast analytics (groupBy + agg)
OPTIMIZE,Compact small files into larger ones for better read performance
ZORDER,Co-locate data by a column so filtered queries skip irrelevant files
VACUUM,"Clean up old, unreferenced files to save storage costs"
Databricks Visualizations,"Create bar, pie, line charts from query results using the built-in UI"
Pipeline Orchestration,Automate Bronze ➔ Silver ➔ Gold with Databricks Workflows


### Congratulations! You have built a complete Medallion Architecture pipeline.

In [1]:
displayHTML("""
<div style="background: #1B3A4B; padding: 35px; border-radius: 12px; margin: 20px 0; font-family: monospace; color: #FFFFFF; border: 2px solid #F9A825;">

<div style="text-align: center; font-size: 20px; font-weight: bold; color: #FF3621; margin-bottom: 25px; letter-spacing: 1px;">YOUR COMPLETE MEDALLION PIPELINE</div>

<div style="text-align: center;">

<div style="display: inline-block; background: #00A972; color: #FFFFFF; padding: 10px 20px; border-radius: 6px; font-size: 12px; margin-bottom: 15px; font-weight: bold;">Source: ADLS Gen2 (CSV files)</div>

<div style="color: #F9A825; font-size: 20px; margin: 8px 0;">&#9660;</div>

<div style="display: inline-block; background: #CD7F32; color: #FFFFFF; padding: 18px 50px; border-radius: 10px; font-weight: bold; margin: 8px; font-size: 16px; border: 3px solid #FFF3E0; box-shadow: 0 4px 12px rgba(205,127,50,0.4);">
BRONZE<br/><span style="font-size: 12px; font-weight: normal;">Raw ingestion into Delta</span><br/><span style="font-size: 11px; color: #FFF3E0;">Lab 04</span>
</div>

<div style="color: #F9A825; font-size: 20px; margin: 8px 0;">&#9660;</div>

<div style="display: inline-block; background: #757575; color: #FFFFFF; padding: 18px 50px; border-radius: 10px; font-weight: bold; margin: 8px; font-size: 16px; border: 3px solid #F5F5F5; box-shadow: 0 4px 12px rgba(117,117,117,0.4);">
SILVER<br/><span style="font-size: 12px; font-weight: normal;">Clean, validate, enrich</span><br/><span style="font-size: 11px; color: #F5F5F5;">Lab 05</span>
</div>

<div style="color: #F9A825; font-size: 20px; margin: 8px 0;">&#9660;</div>

<div style="display: inline-block; background: #F9A825; color: #1B1F24; padding: 18px 50px; border-radius: 10px; font-weight: bold; margin: 8px; font-size: 16px; border: 3px solid #FFFDE7; box-shadow: 0 4px 12px rgba(249,168,37,0.4);">
GOLD<br/><span style="font-size: 12px; font-weight: normal;">Aggregate, optimize, visualize</span><br/><span style="font-size: 11px; color: #1B3A4B;">Lab 06</span>
</div>

<div style="color: #F9A825; font-size: 20px; margin: 8px 0;">&#9660;</div>

<div style="display: inline-block; background: #00A972; color: #FFFFFF; padding: 10px 20px; border-radius: 6px; font-size: 12px; margin-top: 8px; font-weight: bold;">Consumers: Dashboards, BI Tools, ML Models</div>

</div>

<div style="text-align: center; margin-top: 22px; padding-top: 15px; border-top: 1px solid rgba(255,255,255,0.2);">
<span style="color: #F9F7F4;">Orchestrated by: </span><span style="color: #FF3621; font-weight: bold;">Databricks Workflows</span>
<span style="color: #F9F7F4;"> | Schedule: </span><span style="color: #F9A825; font-weight: bold;">Daily 6:00 AM</span>
</div>

</div>
""")

YOUR COMPLETE MEDALLION PIPELINE 

 

 Source: ADLS Gen2 (CSV files) 

 ▼ 

 
BRONZE Raw ingestion into Delta Lab 04 
 

 ▼ 

 
SILVER Clean, validate, enrich Lab 05 
 

 ▼ 

 
GOLD Aggregate, optimize, visualize Lab 06 
 

 ▼ 

 Consumers: Dashboards, BI Tools, ML Models 

 

 
 Orchestrated by: Databricks Workflows 
 | Schedule: Daily 6:00 AM

---

## Cleanup

**Run this only after completing ALL Day 2 labs.** This removes all lab data and tables.

In [ ]:
# ============================================================
# FULL CLEANUP - Run this after completing ALL Day 2 labs
# ============================================================
# WARNING: This removes ALL lab data and tables!

# Uncomment to execute:
# spark.sql("DROP DATABASE IF EXISTS bronze CASCADE")
# spark.sql("DROP DATABASE IF EXISTS silver CASCADE")
# spark.sql("DROP DATABASE IF EXISTS gold CASCADE")
#
# paths_to_clean = [
#     f"{BASE_PATH}/bronze",
#     f"{BASE_PATH}/silver",
#     f"{BASE_PATH}/gold",
#     f"{BASE_PATH}/checkpoints"
# ]
# for path in paths_to_clean:
#     try:
#         dbutils.fs.rm(path, recurse=True)
#         print(f"Removed: {path}")
#     except Exception as e:
#         print(f"Note: {path} - {e}")
#
# print("Full cleanup completed! All lab resources removed.")
# print("Storage costs should stop immediately.")

print("To clean up, uncomment the code above and run this cell.")
print("This removes all Bronze, Silver, and Gold layer data.")